# Global Earthquake Analysis — 2025 (M ≥ 4.5)

Bu notebook USGS verilerini aylık önbelleğe alma ve yeniden deneme mekanizmasıyla toplar; mevcut analizlere Temmuz 2025 bölgesel incelemesi, `magType` karşılaştırması ve dünya haritası ekler.

**Önemli:** Notebook README.md dosyasını değiştirmez. Var olan veri dosyalarıyla API'ye yeniden bağlanmadan analiz yapmak için ilk veri toplama hücresinde `USE_EXISTING_RAW = True` kullanılır. Klasörler depo köküne göre tanımlanır.

In [ ]:
from pathlib import Path
import json
import time
import re
import requests
import pandas as pd
import matplotlib.pyplot as plt
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Notebook'u depo kökünden veya notebooks/ klasöründen çalıştırmayı destekler.
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
MONTHLY = RAW / "monthly_2025"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
for folder in (RAW, MONTHLY, PROCESSED, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)
print('Proje klasörü:', ROOT)

## 1. Veri toplama: yeniden deneme ve aylık önbellek

Başarısız ağ istekleri geçici sunucu hatalarında tekrar denenir. Başarılı her ay ayrı JSON dosyasına kaydedilir. İşlem kesilirse tekrar çalıştırıldığında önceden kaydedilen aylar yeniden indirilmez. Mevcut `earthquakes_2025.json` dosyası, internet olmadan analiz için kullanılabilir.

In [ ]:
API_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query'
USE_EXISTING_RAW = True  # True: mevcut ham dosyayı kullan; False: aylık önbellek ile topla

def make_session():
    retry = Retry(total=4, connect=4, read=4, backoff_factor=1,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=['GET'], respect_retry_after_header=True)
    session = requests.Session()
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    return session

def get_earthquakes():
    complete_file = RAW / 'earthquakes_2025.json'
    if USE_EXISTING_RAW and complete_file.exists():
        with complete_file.open(encoding='utf-8') as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            raise ValueError('Ham veri JSON dosyası liste olmalı.')
        print(f'Mevcut ham veri yüklendi: {len(saved)} kayıt')
        return saved

    collected = []
    with make_session() as session:
        for month in range(1, 13):
            monthly_file = MONTHLY / f'earthquakes_2025_{month:02d}.json'
            if monthly_file.exists():
                with monthly_file.open(encoding='utf-8') as f:
                    earthquakes = json.load(f)
                if not isinstance(earthquakes, list):
                    raise ValueError(f'Hatalı aylık önbellek: {monthly_file}')
                print(f'{month:02d}. ay önbellekten: {len(earthquakes)}')
            else:
                end = f'2025-{month + 1:02d}-01' if month < 12 else '2026-01-01'
                params = {'format':'geojson','starttime':f'2025-{month:02d}-01',
                          'endtime':end,'minmagnitude':4.5,
                          'eventtype':'earthquake','limit':20000}
                response = session.get(API_URL, params=params, timeout=(15, 90))
                response.raise_for_status()
                payload = response.json()
                earthquakes = payload['features']
                if not isinstance(earthquakes, list):
                    raise ValueError(f'{month}. ay: özellikler listesi bulunamadı')
                # Atomik yazma: yarım kalmış dosyayı geçerli önbellek sanmayalım.
                temp = monthly_file.with_suffix('.json.tmp')
                temp.write_text(json.dumps(earthquakes, ensure_ascii=False), encoding='utf-8')
                temp.replace(monthly_file)
                print(f'{month:02d}. ay indirildi: {len(earthquakes)}')
                time.sleep(0.3)
            collected.extend(earthquakes)

    # Bir depreme ait birden fazla sorgu kaydı varsa kimliğe göre tekilleştir.
    unique = {item['id']: item for item in collected}
    records = list(unique.values())
    # Tam veri dosyasını yalnızca bütün aylar başarıyla yüklendikten sonra yaz.
    temp = complete_file.with_suffix('.json.tmp')
    temp.write_text(json.dumps(records, ensure_ascii=False), encoding='utf-8')
    temp.replace(complete_file)
    print('Toplam benzersiz deprem:', len(records))
    return records

records = get_earthquakes()

## 2. Temizlik ve kontrol

`magType` alanı kaynak kayıtlardaki raporlanan büyüklük ölçeğidir. Farklı ölçekler birebir eşdeğer olmayabilir; bunları alt gruplar halinde inceleyeceğiz.

In [ ]:
raw_df = pd.json_normalize(records)
required = ['id','properties.mag','properties.place','properties.time','geometry.coordinates']
missing = [name for name in required if name not in raw_df]
if missing:
    raise KeyError(f'USGS verisinde zorunlu alan eksik: {missing}')

df = pd.DataFrame({
    'id': raw_df['id'],
    'magnitude': pd.to_numeric(raw_df['properties.mag'], errors='coerce'),
    'magType': raw_df['properties.magType'] if 'properties.magType' in raw_df else pd.Series(pd.NA, index=raw_df.index),
    'place': raw_df['properties.place'],
    'time': pd.to_datetime(raw_df['properties.time'], unit='ms', utc=True, errors='coerce'),
})
coordinates = raw_df['geometry.coordinates'].apply(
    lambda x: x if isinstance(x, (list, tuple)) and len(x) >= 3 else [None, None, None]
)
for ix, name in enumerate(('longitude','latitude','depth')):
    df[name] = pd.to_numeric(coordinates.apply(lambda x: x[ix]), errors='coerce')

print('Tekrarlı kimlik sayısı:', df.duplicated('id').sum())
df = df.drop_duplicates('id').copy()
print('Eksik değerler:')
print(df.isna().sum().to_string())

valid = (df['time'].notna() & df['time'].dt.year.eq(2025) &
         df['magnitude'].ge(4.5) &
         df['latitude'].between(-90,90) & df['longitude'].between(-180,180) &
         df['depth'].notna())
print('Filtrelenen kayıt:', int((~valid).sum()))
df = df.loc[valid].copy()
df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month

df.to_csv(PROCESSED / 'earthquakes_2025_clean.csv', index=False)
print('Temiz veri boyutu:', df.shape)
print(df[['id','magnitude','magType','place','time']].head().to_string(index=False))

## 3. Aylık dağılım ve Temmuz artışının bölgesel incelemesi

Bölge bilgisi, USGS `place` metninin son virgülden sonraki kısmından elde edilir. Bu bir idari/coğrafi standardizasyon değil, tanımlayıcı metin gruplamasıdır; örneğin `Russia` bir ülkeyi, `Alaska` bir eyaleti temsil eder. Bulgular yorumlanırken bu sınır dikkate alınmalıdır.

In [ ]:
months = ['Ocak','Şubat','Mart','Nisan','Mayıs','Haziran',
          'Temmuz','Ağustos','Eylül','Ekim','Kasım','Aralık']
monthly_counts = df['month'].value_counts().reindex(range(1,13),fill_value=0).sort_index()
print(monthly_counts.rename(index={i+1:month for i,month in enumerate(months)}).to_string())
fig,ax=plt.subplots(figsize=(12,5))
ax.bar(months,monthly_counts.values)
ax.set_title('2025 Aylık Deprem Sayıları (M ≥ 4.5)')
ax.set_ylabel('Deprem sayısı')
ax.tick_params(axis='x',rotation=45)
fig.tight_layout()
fig.savefig(FIGURES/'monthly_earthquakes_2025.png',dpi=180)
plt.show()

def place_region(place):
    if pd.isna(place) or not str(place).strip():
        return 'Konum belirtilmemiş'
    value = str(place).strip()
    return value.rsplit(',', 1)[-1].strip() if ',' in value else value

df['region_label'] = df['place'].map(place_region)
july = df.loc[df['month'].eq(7)].copy()
july_regions = july['region_label'].value_counts()
print('\nTemmuz 2025 - ilk 15 konum etiketi:')
print(july_regions.head(15).to_string())
print(f'\nTemmuz toplamı: {len(july):,}; ilk bölge payı: {july_regions.iloc[0] / len(july):.1%}')

fig,ax=plt.subplots(figsize=(10,6))
july_regions.head(10).sort_values().plot.barh(ax=ax)
ax.set_title('Temmuz 2025: En çok kayıt gelen 10 bölge etiketi')
ax.set_xlabel('Deprem sayısı (M ≥ 4.5)')
ax.set_ylabel('USGS konum etiketleri')
fig.tight_layout()
fig.savefig(FIGURES/'july_2025_regions.png',dpi=180)
plt.show()

## 4. Büyüklük türleri (`magType`) ve derinlik ilişkisi

Önce tüm raporlanan `magType` türlerinin sayılarını inceliyoruz. Daha sonra **en az 30 geçerli kaydı olan** türlerde, tür içi Pearson korelasyonunu hesaplıyoruz. Örneklem büyüklükleri ve büyüklük ölçekleri farklı olduğundan korelasyonlar nedensellik göstermez; gruplar arası doğrudan karşılaştırmalar dikkatle yapılmalıdır.

In [ ]:
df['magType'] = df['magType'].fillna('Bilinmiyor').astype(str)
mag_counts = df['magType'].value_counts()
print('Büyüklük türleri:\n',mag_counts.to_string())
fig,ax=plt.subplots(figsize=(10,5))
mag_counts.sort_values().plot.barh(ax=ax)
ax.set_title('USGS Büyüklük Türü (magType) Dağılımı')
ax.set_xlabel('Kayıt sayısı')
fig.tight_layout()
fig.savefig(FIGURES/'magnitude_type_distribution.png',dpi=180)
plt.show()

overall_corr = df[['magnitude','depth']].corr().loc['magnitude','depth']
print(f'\nTüm türler birlikte Pearson r: {overall_corr:.4f}')
rows=[]
for mag_type,subset in df.groupby('magType'):
    valid_subset = subset[['magnitude','depth']].dropna()
    n = len(valid_subset)
    if n >= 30 and valid_subset['magnitude'].nunique() > 1 and valid_subset['depth'].nunique() > 1:
        corr = valid_subset['magnitude'].corr(valid_subset['depth'])
        rows.append({'magType':mag_type,'n':n,'pearson_r':corr})
correlation_by_type = pd.DataFrame(rows)
if len(correlation_by_type):
    correlation_by_type = correlation_by_type.sort_values('n',ascending=False)
    print('\nTür bazında korelasyonlar:')
    print(correlation_by_type.to_string(index=False,float_format=lambda x:f'{x:.4f}'))
else:
    print('Korelasyon için yeterli gözlem sayısı olan tür bulunamadı.')

fig,ax=plt.subplots(figsize=(10,6))
ax.scatter(df['depth'],df['magnitude'],alpha=0.25,s=12)
ax.set_title('Derinlik ve Raporlanan Büyüklük')
ax.set_xlabel('Derinlik (km)')
ax.set_ylabel('Raporlanan büyüklük (farklı ölçekler)')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES/'depth_magnitude_scatter.png',dpi=180)
plt.show()

## 5. Dünya haritası üzerinde konumlar

`cartopy` kuruluysa kıyı çizgili harita; kurulu değilse koordinat tabanlı nokta grafiği gösterilir. İlk Cartopy çalıştırmasında Natural Earth kıyı çizgileri için internet bağlantısı gerekebilir. Çevrimdışı ortamda hata olması halinde otomatik olarak koordinat grafiğine geçilir.

In [ ]:
try:
    import cartopy.crs as ccrs
    fig = plt.figure(figsize=(15,7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_global()
    ax.coastlines(linewidth=0.6)
    scatter = ax.scatter(df['longitude'],df['latitude'],c=df['magnitude'],
                         transform=ccrs.PlateCarree(), cmap='plasma',
                         s=10,alpha=0.55)
    ax.gridlines(draw_labels=False,alpha=0.2)
    ax.set_title('2025 Depremleri — Dünya Haritası (M ≥ 4.5)')
    fig.colorbar(scatter,ax=ax,shrink=0.7,label='Raporlanan büyüklük')
except Exception as exc:
    plt.close('all')
    print('Harita altlığı kullanılamadı; koordinat grafiğine geçildi:',str(exc)[:160])
    fig,ax = plt.subplots(figsize=(14,7))
    scatter = ax.scatter(df['longitude'],df['latitude'],c=df['magnitude'],
                         cmap='plasma',s=10,alpha=0.5)
    ax.set_xlim(-180,180)
    ax.set_ylim(-90,90)
    ax.set_xlabel('Boylam')
    ax.set_ylabel('Enlem')
    ax.grid(alpha=0.25)
    ax.set_title('2025 Depremleri — Koordinat Dağılımı (Harita altlığı yok)')
    fig.colorbar(scatter,ax=ax,label='Raporlanan büyüklük')
fig.tight_layout()
fig.savefig(FIGURES/'earthquake_world_map_2025.png',dpi=180,bbox_inches='tight')
plt.show()

## Sonuçlar ve dikkat edilmesi gerekenler

- Temmuz 2025 zirvesi bölge etiketleriyle ayrıştırılmıştır; yalnızca zaman grafiğinden mevsimsellik sonucu çıkarılamaz.
- Farklı büyüklük ölçekleri aynı fiziksel niceliği doğrudan aynı yöntemle ölçmeyebilir. `magType` alt grupları bu sınırlılığı görünür kılar.
- Haritadaki nokta kümeleri 2025, M ≥ 4.5 USGS katalog kayıtlarını temsil eder; tüm sismisiteyi temsil etmez.
- USGS kataloğu daha sonra güncellenebileceğinden canlı API'den yeniden çekilen kayıt sayıları değişebilir.
- **README.md bilinçli olarak bu notebook tarafından yazılmaz.** Açıklamalar ayrı dosyadan yönetilmelidir.